# Common Beam

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/image_analysis/common_beam.ipynb)

The imagers restore each plane with the Gaussian fitted to its own PSF. Given `restoring_beam=[major, minor, pa]` (FWHM and position angle in radians, as `BEAM_FIT_PARAMS`), `image_cube_single_field` and `image_continuum_single_field` restore every plane to that beam, as `tclean` does with `restoringbeam`: the model is convolved with the target beam, and the residual with the Gaussian that takes the plane's PSF beam to the target, times the ratio of the beam areas, so it is in Jy per target beam.

`common_beam` gives the beam of `tclean`'s `restoringbeam='common'`, and `correcting_beam` the Gaussian that takes each PSF beam to a target, refusing a target that does not contain every PSF beam. This notebook runs the recipe on a small simulated cube.

## Install AstroVIPER

Installs AstroVIPER only where it is missing.

In [ ]:
import os
from importlib.metadata import PackageNotFoundError, version

try:
    print("Using astroviper version", version("astroviper"))
except PackageNotFoundError:
    os.system("pip install --upgrade astroviper[all]")

## API

In [ ]:
from astroviper.processing_functions.image_analysis import common_beam, correcting_beam

common_beam?

In [ ]:
correcting_beam?

## The common beam and the containment test

`common_beam` follows CASA's `getCommonBeam` as CASA 6.8.0.4 has it (CAS-14719). The first PSF beam of largest area is the common beam when every other beam fits inside it. Otherwise an ellipse is built around it and the last beam that does not fit, grown in steps of 0.1% until both fit, put in place of the largest beam, and the test is repeated. The result is not the smallest ellipse that contains every beam, and it depends on the order of the beams: frequency fastest, then polarization, then time, as CASA visits the planes of an image.

A target contains a PSF beam when the difference of their covariance matrices, `D = S_target - S_psf`, is positive semidefinite (a target narrower than the PSF beam by up to 5e-7 of its width in some direction counts as rounding). `D` is then the covariance of the Gaussian that takes the PSF beam to the target. The cube imager keeps the residual of a plane whose PSF beam the target does not contain unconvolved and unscaled, as `tclean` does, and warns; the continuum imager refuses such a target, and any target when its PSF fit failed, right after that fit. `correcting_beam` raises, so the check can run before the imaging run.

## A small simulated cube

The 27 antennas of the VLA in its D configuration observe a 1 Jy point source at the phase centre for 6 hours, in 8 channels from 2 to 3.75 GHz, so the PSF beam shrinks with frequency. The data go to a temporary directory.

In [ ]:
%matplotlib inline

import shutil
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from astropy.coordinates import SkyCoord
from matplotlib.patches import Ellipse
from xradio.measurement_set import open_processing_set

import astroviper.distributed_applications as distributed_applications
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout

ARCSEC = np.pi / (180 * 3600)
work = tempfile.mkdtemp(prefix="common_beam_")
direction = SkyCoord(ra="19h59m28.5s", dec="+40d44m01.5s", frame="fk5")
phase_center = np.array([direction.ra.rad, direction.dec.rad])
antenna_xds = read_telescope_layout("vla.d")
ps_store = distributed_applications.simulation.simulate_processing_set(
    ps_store=os.path.join(work, "point_source.ps.zarr"),
    antenna_xds=antenna_xds,
    time_params={
        "time_start": "2019-10-03T16:00:00.000",
        "time_delta": 1800.0,
        "n_samples": 12,
    },
    frequency_params={
        "freq_start": 2e9,
        "freq_delta": 0.25e9,
        "n_channels": 8,
        "channel_width": 1e7,
    },
    polarization=["RR", "LL"],
    point_source_flux=np.array([[[[1.0, 0.0, 0.0, 1.0]]]]),
    point_source_ra_dec=phase_center[None, None, :],
    phase_center_ra_dec=phase_center[None, :],
    beam_models=[airy_disk_model("vla_nvss")],
    beam_model_map=np.zeros(antenna_xds.sizes["antenna_name"], dtype=int),
    n_frequency_chunks=2,
    overwrite=True,
)["ps_store"]

## The PSF beams of the imager

`point_spread_function_cube_single_field` makes the PSF of every channel as the imager does, with the imager's `image_params` and weighting, and fits the same Gaussian. It is one more pass over the data, which loads the weights but not the visibilities.

In [ ]:
image_params = {
    "image_size": [128, 128],
    "cell_size": np.array([-5.0, 5.0]) * ARCSEC,
    "phase_direction": phase_center,
    "frequency_coords": open_processing_set(ps_store).xr_ps.get_freq_axis().values,
    "polarization_coords": ["I", "V"],
    "time_coords": [0],
    "fft_padding": 1.2,
}
imaging_weights_params = {"weighting": "briggs", "robust": 0.5}
psf = distributed_applications.imaging.point_spread_function_cube_single_field(
    ps_store,
    image_params,
    imaging_weights_params,
    instrument_polarization_basis="circular",
    processing_set_data_group_name="base",
    n_mapping_parallelism={"frequency": 2},
)
frequency_ghz = pd.Index(psf.frequency.values / 1e9, name="frequency (GHz)")


def beam_table(beams, index=None):
    """Beams [major, minor, pa] in radians as a table in arcsec and degrees."""
    beams = np.reshape(beams, (-1, 3))
    return pd.DataFrame(
        {
            "major (arcsec)": beams[:, 0] / ARCSEC,
            "minor (arcsec)": beams[:, 1] / ARCSEC,
            "pa (deg)": np.degrees(beams[:, 2]),
        },
        index=index,
    )


beams = psf["BEAM_FIT_PARAMS_POINT_SPREAD_FUNCTION"].isel(polarization=[0])
beam_table(beams.values, frequency_ghz)

## The common beam, checked before the run

The restore applies the PSF beam of the first polarization, here Stokes I, to every polarization, so the common beam is that of the Stokes I beams. `correcting_beam` raises if the target does not contain a plane's PSF beam. Here the largest beam contains the others, so the common beam is the beam of the lowest channel, its position angle in (-90, 90] deg where the fit gives [0, 180) deg. The correcting Gaussian of that channel is zero and its area ratio 1, so its residual is left as it is.

In [ ]:
target = common_beam(beams)
correcting, area_ratio = correcting_beam(beams, target)
print(beam_table(target, ["common beam"]).to_string(float_format="{:.3f}".format))
beam_table(correcting, frequency_ghz).assign(**{"area ratio": area_ratio.ravel()})

## A rounded target

A target typed from a printout can be smaller than the largest PSF beam along some direction. Here the common beam is rounded down to 0.1 arcsec and 0.1 deg: `correcting_beam` names the planes whose PSF beam it does not contain and the factor by which it would have to grow. Round such a beam up.

In [ ]:
units = np.array([ARCSEC, ARCSEC, np.pi / 180])
rounded_down = np.floor(target / units * 10) / 10 * units
try:
    correcting_beam(beams, rounded_down)
except ValueError as error:
    print(error)

## Restoring the cube to the common beam

`restoring_beam` takes the target. With `restore=True` the imager restores every channel to it and records it as `BEAM_FIT_PARAMS_SKY_RESTORED`, the beam of the restored group; `SKY_RESIDUAL` is not changed. For comparison, a second run restores each channel with its own PSF beam.

In [ ]:
cycle_control_params = {
    "max_iter": 200,
    "max_cycles": 3,
    "threshold": 0.001,
    "gain": 0.1,
    "psf_sidelobe_factor": 1.5,
    "max_iter_per_cycle": -1,
    "min_psf_fraction": 0.05,
    "max_psf_fraction": 0.8,
}
restored = {}
for name, restoring_beam in (("own PSF beam", None), ("common beam", target)):
    image_store = os.path.join(work, name.replace(" ", "_") + ".img.zarr")
    distributed_applications.imaging.image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params=imaging_weights_params,
        cycle_control_params=cycle_control_params,
        instrument_polarization_basis="circular",
        processing_set_data_group_name="base",
        image_data_variables_keep=["sky_residual", "sky_model"],
        n_mapping_parallelism={"frequency": 2},
        restore=True,
        restoring_beam=restoring_beam,
        overwrite=True,
    )
    restored[name] = xr.open_zarr(image_store)

restored["common beam"].attrs["data_groups"]["restored"]

In [ ]:
beam_table(
    restored["common beam"]
    .BEAM_FIT_PARAMS_SKY_RESTORED.isel(time=0, polarization=0)
    .values,
    frequency_ghz,
)

The restored point source in the lowest and the highest channel: its half-maximum contour (wide, blue) lies on the beam it was restored to (red, dashed). Restored to the common beam, the point source has the same shape in every channel, and its peak stays 1 Jy/beam.

In [ ]:
def draw_beam(axis, beam, **style):
    """The FWHM ellipse of a beam [major, minor, pa] (radians), east to the left."""
    axis.add_patch(
        Ellipse(
            (0.0, 0.0),
            beam[0] / ARCSEC,
            beam[1] / ARCSEC,
            angle=90.0 - np.degrees(beam[2]),
            fill=False,
            **style,
        )
    )


figure, axes = plt.subplots(2, 2, figsize=(9, 9), constrained_layout=True)
for row, (name, image) in zip(axes, restored.items(), strict=True):
    for axis, channel in zip(row, (0, -1), strict=True):
        plane = image.SKY_RESTORED.isel(time=0, polarization=0, frequency=channel)
        l_arcsec, m_arcsec = plane.l.values / ARCSEC, plane.m.values / ARCSEC
        axis.pcolormesh(
            l_arcsec, m_arcsec, plane.values.T, cmap="gray_r", vmin=-0.05, vmax=1.0
        )
        axis.contour(
            l_arcsec,
            m_arcsec,
            plane.values.T,
            levels=[0.5],
            colors="tab:blue",
            linewidths=4,
            alpha=0.5,
        )
        beam = target if name == "common beam" else beams.values[channel, 0]
        draw_beam(axis, beam, color="tab:red", linestyle="--", linewidth=1.5, zorder=3)
        axis.set(
            xlim=(150, -150),
            ylim=(-150, 150),
            aspect="equal",
            xlabel="l (arcsec)",
            ylabel="m (arcsec)",
            title=f"{name}, {float(plane.frequency) / 1e9:.2f} GHz",
        )

pd.DataFrame(
    {
        name: image.SKY_RESTORED.isel(time=0, polarization=0).max(dim=["l", "m"]).values
        for name, image in restored.items()
    },
    index=frequency_ghz,
).rename_axis(columns="peak (Jy/beam)")

## When no beam contains the others

When the PSF beams differ in orientation, the largest does not contain the others, and `common_beam` builds and grows the enclosing ellipse as CASA does. It contains every beam here, which `correcting_beam` confirms, but it is not the smallest ellipse that would.

In [ ]:
differing = np.array([(3.0, 1.0, 20.0), (2.8, 1.1, 60.0), (2.5, 1.0, 40.0)]) * units
enclosing = common_beam(differing[np.newaxis, :, np.newaxis])
correcting_beam(differing, enclosing)

figure, axis = plt.subplots(figsize=(4.5, 4.5), constrained_layout=True)
for beam in differing:
    draw_beam(axis, beam, color="tab:blue")
draw_beam(axis, enclosing, color="tab:red", linestyle="--", linewidth=2)
axis.set(
    xlim=(2.2, -2.2),
    ylim=(-2.2, 2.2),
    aspect="equal",
    xlabel="east (arcsec)",
    ylabel="north (arcsec)",
)
beam_table(
    np.vstack([differing, enclosing]), ["beam 0", "beam 1", "beam 2", "common beam"]
)

In [ ]:
shutil.rmtree(work)